# Experimental Validation — Consistent Normalization

Matches the exact normalization from `Write_dataset_4ch.ipynb`:
- Ch0: `c [mol/m³] / 6000`
- Ch1: `(t - t_min) / (t_max - t_min)` using COMSOL time grid
- Ch2: `diff(c_norm, axis=0) / dx_comsol`, then `/abs_max`
- Ch3: `diff(c_norm, axis=1) / dt_comsol`, then `/abs_max`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.interpolate import RegularGridInterpolator
from scipy.special import comb


## 1. Load COMSOL Grid Info
We need the exact dx, dt, time channel from training.


In [ ]:
# *** UPDATE THESE PATHS ***
TRAIN_DATA_PATH = '/Users/ZirongHe/Desktop/C242 Machine Learning/Final project/fixed_rho_data_4ch.pt'
MODEL_PATH = '/Users/ZirongHe/Desktop/C242 Machine Learning/Final project/phase2_4ch_steady_weighted.pt'
YMEAN_PATH = '/Users/ZirongHe/Desktop/C242 Machine Learning/Final project/y_mean.npy'
YSTD_PATH  = '/Users/ZirongHe/Desktop/C242 Machine Learning/Final project/y_std.npy'
EXP_CSV    = '/Users/ZirongHe/Desktop/C242 Machine Learning/Final project/1 velocity and concentration.csv'

# Dataset class needed for unpickling
from torch.utils.data import Dataset
import glob, os

class MultiFileSpatioTemporalDataset_4ch(Dataset):
    def __init__(self, folder_path, C_max=6000.0):
        file_list = sorted(glob.glob(os.path.join(folder_path, "batch_*_conc_fix_rho.csv")))
        if not file_list: raise ValueError(f"No CSV files found in {folder_path}")
        with open(file_list[0], 'r') as f:
            skip_idx = 0
            for i, line in enumerate(f):
                if line.startswith('% X,t'): skip_idx = i; break
        sample_df = pd.read_csv(file_list[0], sep=',', skiprows=skip_idx)
        sample_df.rename(columns={sample_df.columns[0]: 'x'}, inplace=True)
        if sample_df['x'].dtype == object:
            sample_df['x'] = sample_df['x'].astype(str).str.replace('% ', '').astype(float)
        x_unique, t_unique = np.sort(sample_df['x'].unique()), np.sort(sample_df['t'].unique())
        self.N_x, self.N_t = len(x_unique), len(t_unique)
        self.x, self.t = torch.tensor(x_unique, dtype=torch.float32), torch.tensor(t_unique, dtype=torch.float32)
        self.dx, self.dt, self.C_max = np.diff(x_unique), np.diff(t_unique), C_max
        t_arr = t_unique.astype(np.float64)
        t_linear = (t_arr - t_arr.min()) / (t_arr.max() - t_arr.min() + 1e-12)
        self._time_channel = np.tile(t_linear, (self.N_x, 1)).astype(np.float32)
        all_c, all_dcdx, all_dcdt, all_targets = [], [], [], []
        for file in file_list:
            with open(file, 'r') as f:
                for i, line in enumerate(f):
                    if line.startswith('% X,t'): skip_idx = i; break
            df2 = pd.read_csv(file, sep=',', skiprows=skip_idx)
            c_cols = [col for col in df2.columns if col.startswith('c (mol/m^3) @')]
            c_raw = df2[c_cols].values.T.reshape(-1, self.N_x, self.N_t)
            for c in c_raw:
                c_norm = (c / C_max).astype(np.float32)
                dcdx = np.diff(c_norm, axis=0) / self.dx[:, None].astype(np.float32)
                dcdx = np.concatenate([dcdx, dcdx[-1:, :]], axis=0)
                am = np.abs(dcdx).max()
                if am > 1e-10: dcdx = dcdx / am
                dcdt = np.diff(c_norm, axis=1) / self.dt[None, :].astype(np.float32)
                dcdt = np.concatenate([dcdt, dcdt[:, -1:]], axis=1)
                am = np.abs(dcdt).max()
                if am > 1e-10: dcdt = dcdt / am
                all_c.append(c_norm); all_dcdx.append(dcdx.astype(np.float32)); all_dcdt.append(dcdt.astype(np.float32))
            for col in c_cols: all_targets.append(self._parse_targets(col))
            del df2
        self.c_grids, self.dcdx, self.dcdt = np.array(all_c), np.array(all_dcdx), np.array(all_dcdt)
        self.Total_S = len(all_c)
        self.targets = {
            'rho': np.array([t['rho'] for t in all_targets], dtype=np.float32),
            'Tf': np.array([t['Tf'] for t in all_targets], dtype=np.float32),
            'D': np.array([t['D'] for t in all_targets], dtype=np.float32),
            'kappa': np.array([t['kappa'] for t in all_targets], dtype=np.float32),
            't_param': np.array([t['t'] for t in all_targets], dtype=np.float32),
        }
    def _parse_targets(self, header_string):
        param_chunk = header_string.split('@')[1].strip()
        parsed = {'rho': [], 'Tf': [], 'D': [], 'kappa': [], 't': []}
        for pair in param_chunk.split(';'):
            if '=' in pair:
                k, v = pair.split('='); base = k.strip().split('_')[0]
                if base in parsed: parsed[base].append(float(v.strip()))
        return parsed
    def __len__(self): return self.Total_S
    def __getitem__(self, idx):
        inp = torch.tensor(np.stack([self.c_grids[idx], self._time_channel, self.dcdx[idx], self.dcdt[idx]]), dtype=torch.float32)
        return inp, {k: torch.tensor(v[idx]) for k, v in self.targets.items()}

# Load training dataset to extract grid info
train_ds = torch.load(TRAIN_DATA_PATH, map_location='cpu', weights_only=False)

x_comsol = train_ds.x.numpy()       # (581,) in COMSOL units (μm)
t_comsol = train_ds.t.numpy()       # (121,) in seconds
dx_comsol = train_ds.dx             # (580,) 
dt_comsol = train_ds.dt             # (120,)
C_max = train_ds.C_max              # 6000.0
time_channel = train_ds._time_channel  # (581, 121) precomputed
N_x_comsol, N_t_comsol = train_ds.N_x, train_ds.N_t

print(f"COMSOL grid: {N_x_comsol} × {N_t_comsol}")
print(f"x range: [{x_comsol[0]:.2f}, {x_comsol[-1]:.2f}] (COMSOL units)")
print(f"t range: [{t_comsol[0]:.2f}, {t_comsol[-1]:.2f}] s")
print(f"dx[0]: {dx_comsol[0]:.4f}, dt[0]: {dt_comsol[0]:.4f}")
print(f"C_max: {C_max}")

# Print training data channel ranges for comparison
sample_inp, _ = train_ds[0]
print(f"\nTraining sample Ch0 range: [{sample_inp[0].min():.4f}, {sample_inp[0].max():.4f}]")
print(f"Training sample Ch2 range: [{sample_inp[2].min():.4f}, {sample_inp[2].max():.4f}]")
print(f"Training sample Ch3 range: [{sample_inp[3].min():.4f}, {sample_inp[3].max():.4f}]")


## 2. Load & Reshape Experimental Data


In [ ]:
x_exp = np.sort(df['x'].unique())
t_exp = np.sort(df['t'].unique())
N_x_exp, N_t_exp = len(x_exp), len(t_exp)

c_grid_exp = np.zeros((N_x_exp, N_t_exp), dtype=np.float64)
for j, t_val in enumerate(t_exp):
    df_t = df[df['t'] == t_val].sort_values('x')
    c_grid_exp[:, j] = df_t['c'].values

# Flip x to match COMSOL convention
c_grid_exp = c_grid_exp[::-1, :].copy()

# Cut off edge effects: keep x in [0.05, 0.95]
X_CUTOFF_MIN, X_CUTOFF_MAX = 0.05, 0.95
edge_mask = (x_exp >= X_CUTOFF_MIN) & (x_exp <= X_CUTOFF_MAX)
x_exp = x_exp[edge_mask]
c_grid_exp = c_grid_exp[edge_mask, :]
N_x_exp = len(x_exp)

print(f"Grid after edge cutoff: {N_x_exp} × {N_t_exp}")
print(f"x range: [{x_exp.min():.4f}, {x_exp.max():.4f}]")
print(f"c range: [{c_grid_exp.min():.4f}, {c_grid_exp.max():.4f}] mol/L")

# Verify: plot before/after
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
for j in [0, N_t_exp//4, N_t_exp//2, -1]:
    ax.plot(x_exp, c_grid_exp[:, j], label=f't={t_exp[j]:.0f}s')
ax.set_xlabel('x'); ax.set_ylabel('c [mol/L]')
ax.set_title(f'After edge cutoff [{X_CUTOFF_MIN}, {X_CUTOFF_MAX}]')
ax.legend(); plt.tight_layout(); plt.show()


## 3. Interpolate to COMSOL Grid


In [ ]:
# Normalize both grids to [0, 1]
x_exp_n = (x_exp - x_exp.min()) / (x_exp.max() - x_exp.min())
t_exp_n = (t_exp - t_exp.min()) / (t_exp.max() - t_exp.min())
x_com_n = (x_comsol - x_comsol.min()) / (x_comsol.max() - x_comsol.min())
t_com_n = (t_comsol - t_comsol.min()) / (t_comsol.max() - t_comsol.min())

interp = RegularGridInterpolator((x_exp_n, t_exp_n), c_grid_exp,
                                  method='linear', bounds_error=False, fill_value=None)
xx, tt = np.meshgrid(x_com_n, t_com_n, indexing='ij')
c_interp = interp(np.stack([xx.ravel(), tt.ravel()], axis=-1)).reshape(N_x_comsol, N_t_comsol)

print(f"Interpolated: {c_interp.shape}")
print(f"c range: [{c_interp.min():.4f}, {c_interp.max():.4f}] mol/L")


## 4. Build 4 Channels (Exact Same as Training)


In [ ]:
# Ch0: c [mol/m³] / C_max — EXACTLY as in Write_dataset_4ch
c_mol_m3 = c_interp * 1000.0   # mol/L → mol/m³
c_norm = (c_mol_m3 / C_max).astype(np.float32)

# Ch1: time channel — use the EXACT precomputed one from training
# (already linear-scaled using COMSOL t_min/t_max)
ch_time = time_channel.astype(np.float32)

# Ch2: dcdx — use COMSOL dx, same formula as Write_dataset_4ch
dcdx = np.diff(c_norm, axis=0) / dx_comsol[:, None].astype(np.float32)
dcdx = np.concatenate([dcdx, dcdx[-1:, :]], axis=0)
am = np.abs(dcdx).max()
if am > 1e-10: dcdx = dcdx / am
dcdx = dcdx.astype(np.float32)

# Ch3: dcdt — use COMSOL dt, same formula as Write_dataset_4ch
dcdt = np.diff(c_norm, axis=1) / dt_comsol[None, :].astype(np.float32)
dcdt = np.concatenate([dcdt, dcdt[:, -1:]], axis=1)
am = np.abs(dcdt).max()
if am > 1e-10: dcdt = dcdt / am
dcdt = dcdt.astype(np.float32)

X_exp = np.stack([c_norm, ch_time, dcdx, dcdt]).astype(np.float32)

print(f"X shape: {X_exp.shape}")
print(f"Ch0 (c):    [{c_norm.min():.4f}, {c_norm.max():.4f}]")
print(f"Ch1 (t):    [{ch_time.min():.4f}, {ch_time.max():.4f}]")
print(f"Ch2 (dcdx): [{dcdx.min():.4f}, {dcdx.max():.4f}]")
print(f"Ch3 (dcdt): [{dcdt.min():.4f}, {dcdt.max():.4f}]")

# Compare with training data
print(f"\n--- Compare with training sample 0 ---")
s0, _ = train_ds[0]
for ch, name in enumerate(['c', 't', 'dcdx', 'dcdt']):
    print(f"  {name}: train=[{s0[ch].min():.4f}, {s0[ch].max():.4f}]  exp=[{X_exp[ch].min():.4f}, {X_exp[ch].max():.4f}]")

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
titles = ['c(x,t)', 't', '∂c/∂x', '∂c/∂t']
cmaps = ['viridis', 'viridis', 'RdBu_r', 'RdBu_r']
for ch in range(4):
    im = axes[0, ch].imshow(X_exp[ch], aspect='auto', cmap=cmaps[ch])
    axes[0, ch].set_title(f'Exp {titles[ch]}'); plt.colorbar(im, ax=axes[0, ch], fraction=0.046)
    im2 = axes[1, ch].imshow(s0[ch].numpy(), aspect='auto', cmap=cmaps[ch])
    axes[1, ch].set_title(f'Train sample 0 {titles[ch]}'); plt.colorbar(im2, ax=axes[1, ch], fraction=0.046)
plt.suptitle('Experimental (top) vs Training sample (bottom)', fontsize=13)
plt.tight_layout(); plt.show()


## 5. Predict


In [ ]:
class CNN_TwoHead(nn.Module):
    def __init__(self, in_channels=4, n_D=4, n_tp=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 5, padding=2), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.D_head = nn.Sequential(nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, n_D))
        self.tp_head = nn.Sequential(nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3),
                                     nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, n_tp))
    def forward(self, x):
        feat = self.features(x)
        return torch.cat([self.D_head(feat), self.tp_head(feat)], dim=1)

model = CNN_TwoHead(in_channels=4)
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()

y_mean = np.load(YMEAN_PATH)
y_std = np.load(YSTD_PATH)
print(f"y_mean: {y_mean.round(4)}")
print(f"y_std:  {y_std.round(4)}")

# Predict
X_tensor = torch.tensor(X_exp).unsqueeze(0)
with torch.no_grad():
    pred_norm = model(X_tensor).numpy().squeeze()

pred = pred_norm * y_std + y_mean
D_pred = pred[:4]
tp_pred = pred[4:]

print(f"\nCNN predictions (Bernstein coefficients):")
print(f"  D:   {D_pred.round(4)}")
print(f"  t⁺⁰: {tp_pred.round(4)}")

ok = np.all((pred >= -0.2) & (pred <= 1.2))
print(f"  {'✓' if ok else '⚠'} Coefficients {'in' if ok else 'OUT OF'} reasonable range")


## 6. Compare with Experimental Measurements


In [ ]:
# Raw electrochemical measurements
c_meas = np.array([0.25, 0.47, 0.87, 1.20, 1.59, 1.87, 2.11, 2.38,
                    2.58, 2.76, 3.05, 3.36, 3.49, 3.78])  # mol/L

D_meas = np.array([6.0e-8, 7.8e-8, 1.0e-7, 1.3e-7, 1.1e-7, 8.4e-8, 7.0e-8,
                    5.8e-8, 9.4e-8, 9.0e-8, 6.5e-8, 6.3e-8, 5.9e-8, 4.2e-8])  # cm²/s

tp_meas = np.array([0.07, 0.23, 0.40, 0.33, 0.43, 0.20, 0.08, -0.08,
                     -0.38, 0.10, 0.41, 0.33, 0.18, -0.02])

# Normalize to CNN basis
lnD_min, lnD_max = -27.63102112, -24.63528884
tp_min, tp_max = -0.25, 1.0
C_MIN_NORM, C_MAX_NORM = 0.0, 6.0

c_meas_norm = (c_meas - C_MIN_NORM) / (C_MAX_NORM - C_MIN_NORM)
D_meas_norm = (np.log(D_meas * 1e-4) - lnD_min) / (lnD_max - lnD_min)
tp_meas_norm = (tp_meas - tp_min) / (tp_max - tp_min)

# Bernstein curves from CNN prediction
def bernstein_poly(coeffs, t):
    n = len(coeffs) - 1
    return sum(coeffs[k] * comb(n,k) * (t**k) * ((1-t)**(n-k)) for k in range(n+1))

c_eval = np.linspace(0, 1, 200)
D_curve = bernstein_poly(D_pred, c_eval)
tp_curve = bernstein_poly(tp_pred, c_eval)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(c_meas_norm, D_meas_norm, color='red', s=80, zorder=5,
                edgecolors='darkred', label='Electrochemical measurements')
axes[0].plot(c_eval, D_curve, 'b-', lw=2.5, label='CNN prediction')
axes[0].axvspan(0.3, 0.7, alpha=0.08, color='green', label='Observable range')
axes[0].set_xlabel('Normalized concentration', fontsize=12)
axes[0].set_ylabel('D (normalized)', fontsize=12)
axes[0].set_title('Diffusivity', fontsize=13); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].scatter(c_meas_norm, tp_meas_norm, color='red', s=80, zorder=5,
                edgecolors='darkred', label='Electrochemical measurements')
axes[1].plot(c_eval, tp_curve, 'b-', lw=2.5, label='CNN prediction')
axes[1].axvspan(0.3, 0.7, alpha=0.08, color='green', label='Observable range')
axes[1].set_xlabel('Normalized concentration', fontsize=12)
axes[1].set_ylabel('t⁺⁰ (normalized)', fontsize=12)
axes[1].set_title('Transference number', fontsize=13); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN Prediction vs Experimental Measurements', fontsize=14)
plt.tight_layout(); plt.show()

print(f"CNN D coefficients:  {D_pred.round(4)}")
print(f"CNN t⁺⁰ coefficients: {tp_pred.round(4)}")
